# Credit Card Default Prediction with Support Vector Machines

This notebook analyses the **UCI Default of Credit Card Clients** dataset and compares two Support Vector Machine classifiers:

- Linear SVM
- RBF-kernel SVM

The original project has been cleaned up to improve **reproducibility**, **compatibility with current scikit-learn versions**, and **cross-validation methodology**.

### Main improvements

- repository-relative dataset loading
- stratified train/test splits
- preprocessing inside a scikit-learn `Pipeline`
- no preprocessing leakage across cross-validation folds
- `StratifiedKFold` instead of plain `KFold`
- modern `ConfusionMatrixDisplay`
- reproducible random seeds


## 1. Imports

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.model_selection import (
    train_test_split,
    GridSearchCV,
    StratifiedKFold,
)
from sklearn.svm import LinearSVC, SVC
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    ConfusionMatrixDisplay,
)

RANDOM_STATE = 42

## 2. Load the dataset

The notebook looks for an `.xls` or `.xlsx` file inside the repository's `data/` directory, so it does not depend on a local Windows path.

> If reading the `.xls` file raises an engine error, install `xlrd` with `pip install xlrd`.


In [ ]:
DATA_DIR = Path("data")

excel_files = sorted(DATA_DIR.glob("*.xls")) + sorted(DATA_DIR.glob("*.xlsx"))

if not excel_files:
    raise FileNotFoundError(
        "No Excel dataset found in the 'data' directory. "
        "Place the UCI credit-card default dataset inside data/."
    )

DATA_PATH = excel_files[0]
print(f"Loading dataset from: {DATA_PATH}")

# The original UCI Excel file has a title row before the actual header.
df = pd.read_excel(DATA_PATH, header=1)

df.head()

## 3. Basic cleaning

In [ ]:
# Remove the customer identifier because it is not a predictive feature.
if "ID" in df.columns:
    df = df.drop(columns="ID")

# Ensure all model columns are numeric.
df = df.apply(pd.to_numeric, errors="raise")

print("Shape:", df.shape)
df.head()

In [ ]:
TARGET = "default payment next month"

if TARGET not in df.columns:
    raise KeyError(
        f"Target column '{TARGET}' was not found. "
        f"Available columns: {list(df.columns)}"
    )

print("Missing values:", int(df.isna().sum().sum()))
print("\nTarget distribution:")
print(df[TARGET].value_counts().sort_index())

### Undefined categorical codes

In the original analysis, rows where `EDUCATION == 0` or `MARRIAGE == 0` were removed because those codes are not part of the documented categories used by the project.


In [ ]:
before = len(df)

df_clean = df.loc[
    (df["EDUCATION"] != 0) &
    (df["MARRIAGE"] != 0)
].copy()

removed = before - len(df_clean)

print(f"Rows before cleaning: {before}")
print(f"Rows removed: {removed}")
print(f"Rows after cleaning: {len(df_clean)}")

## 4. Inspect class balance

In [ ]:
class_counts = (
    df_clean[TARGET]
    .value_counts()
    .sort_index()
    .rename(index={0: "Did not default", 1: "Defaulted"})
)

ax = class_counts.plot(kind="bar")
ax.set_title("Target class distribution")
ax.set_xlabel("Class")
ax.set_ylabel("Number of observations")
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

class_counts

## 5. Balanced modelling sample

The original project compared the models on a balanced subset. To preserve that design while keeping the process reproducible, we take **1,000 observations from each class without replacement**.

This means the final test accuracy describes performance on this **balanced experimental sample**, not on the original real-world class distribution.


In [ ]:
N_PER_CLASS = 1000

class_0 = df_clean[df_clean[TARGET] == 0]
class_1 = df_clean[df_clean[TARGET] == 1]

if min(len(class_0), len(class_1)) < N_PER_CLASS:
    raise ValueError("There are not enough observations to sample 1,000 rows per class.")

balanced_df = pd.concat(
    [
        class_0.sample(n=N_PER_CLASS, random_state=RANDOM_STATE),
        class_1.sample(n=N_PER_CLASS, random_state=RANDOM_STATE),
    ],
    ignore_index=True,
).sample(frac=1, random_state=RANDOM_STATE).reset_index(drop=True)

balanced_df[TARGET].value_counts().sort_index()

## 6. Train/test split

In [ ]:
X = balanced_df.drop(columns=TARGET)
y = balanced_df[TARGET].astype(int)

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=RANDOM_STATE,
    stratify=y,
)

print("Training shape:", X_train.shape)
print("Test shape:", X_test.shape)
print("\nTraining target distribution:")
print(y_train.value_counts().sort_index())
print("\nTest target distribution:")
print(y_test.value_counts().sort_index())

## 7. Preprocessing pipeline

Categorical variables are one-hot encoded and continuous variables are standardized.

Crucially, preprocessing is placed **inside the pipeline**. During cross-validation, the encoder and scaler are fitted separately on each training fold, preventing preprocessing leakage.


In [ ]:
categorical_features = [
    "SEX",
    "EDUCATION",
    "MARRIAGE",
    "PAY_0",
    "PAY_2",
    "PAY_3",
    "PAY_4",
    "PAY_5",
    "PAY_6",
]

categorical_features = [c for c in categorical_features if c in X.columns]
numeric_features = [c for c in X.columns if c not in categorical_features]

preprocessor = ColumnTransformer(
    transformers=[
        (
            "categorical",
            OneHotEncoder(handle_unknown="ignore"),
            categorical_features,
        ),
        (
            "numeric",
            StandardScaler(),
            numeric_features,
        ),
    ]
)

inner_cv = StratifiedKFold(
    n_splits=3,
    shuffle=True,
    random_state=RANDOM_STATE,
)

outer_cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=RANDOM_STATE,
)

## 8. Linear SVM — hyperparameter tuning

In [ ]:
linear_pipeline = Pipeline(
    steps=[
        ("preprocess", preprocessor),
        (
            "model",
            LinearSVC(
                dual="auto",
                max_iter=10000,
                random_state=RANDOM_STATE,
            ),
        ),
    ]
)

linear_param_grid = {
    "model__C": [0.01, 0.1, 0.5, 1, 10, 100],
}

linear_grid = GridSearchCV(
    estimator=linear_pipeline,
    param_grid=linear_param_grid,
    scoring="accuracy",
    cv=inner_cv,
    n_jobs=-1,
    refit=True,
)

linear_grid.fit(X_train, y_train)

print("Best parameters:", linear_grid.best_params_)
print(f"Best inner-CV accuracy: {linear_grid.best_score_:.3f}")

### Linear SVM — test-set evaluation

In [ ]:
linear_pred = linear_grid.predict(X_test)

print(f"Test accuracy: {accuracy_score(y_test, linear_pred):.3f}\n")
print(
    classification_report(
        y_test,
        linear_pred,
        target_names=["Did not default", "Defaulted"],
        digits=3,
    )
)

ConfusionMatrixDisplay.from_predictions(
    y_test,
    linear_pred,
    display_labels=["Did not default", "Defaulted"],
    values_format="d",
)
plt.title("Linear SVM — Confusion Matrix")
plt.tight_layout()
plt.show()

## 9. Linear SVM — nested cross-validation

In [ ]:
linear_outer_scores = []
linear_outer_params = []

for fold, (train_idx, valid_idx) in enumerate(outer_cv.split(X, y), start=1):
    X_outer_train = X.iloc[train_idx]
    X_outer_valid = X.iloc[valid_idx]
    y_outer_train = y.iloc[train_idx]
    y_outer_valid = y.iloc[valid_idx]

    search = GridSearchCV(
        estimator=linear_pipeline,
        param_grid=linear_param_grid,
        scoring="accuracy",
        cv=inner_cv,
        n_jobs=-1,
        refit=True,
    )
    search.fit(X_outer_train, y_outer_train)

    pred = search.predict(X_outer_valid)
    score = accuracy_score(y_outer_valid, pred)

    linear_outer_scores.append(score)
    linear_outer_params.append(search.best_params_)

    print(
        f"Fold {fold}: accuracy={score:.3f}, "
        f"best C={search.best_params_['model__C']}"
    )

print(
    f"\nNested CV accuracy: "
    f"{np.mean(linear_outer_scores):.3f} "
    f"(± {np.std(linear_outer_scores):.3f})"
)

## 10. RBF SVM — hyperparameter tuning

In [ ]:
rbf_pipeline = Pipeline(
    steps=[
        ("preprocess", preprocessor),
        (
            "model",
            SVC(
                kernel="rbf",
                random_state=RANDOM_STATE,
            ),
        ),
    ]
)

rbf_param_grid = {
    "model__C": [0.1, 0.5, 1, 10, 100],
    "model__gamma": ["scale", 0.001, 0.01, 0.1, 1],
}

rbf_grid = GridSearchCV(
    estimator=rbf_pipeline,
    param_grid=rbf_param_grid,
    scoring="accuracy",
    cv=inner_cv,
    n_jobs=-1,
    refit=True,
)

rbf_grid.fit(X_train, y_train)

print("Best parameters:", rbf_grid.best_params_)
print(f"Best inner-CV accuracy: {rbf_grid.best_score_:.3f}")

### RBF SVM — test-set evaluation

In [ ]:
rbf_pred = rbf_grid.predict(X_test)

print(f"Test accuracy: {accuracy_score(y_test, rbf_pred):.3f}\n")
print(
    classification_report(
        y_test,
        rbf_pred,
        target_names=["Did not default", "Defaulted"],
        digits=3,
    )
)

ConfusionMatrixDisplay.from_predictions(
    y_test,
    rbf_pred,
    display_labels=["Did not default", "Defaulted"],
    values_format="d",
)
plt.title("RBF SVM — Confusion Matrix")
plt.tight_layout()
plt.show()

## 11. RBF SVM — nested cross-validation

In [ ]:
rbf_outer_scores = []
rbf_outer_params = []

for fold, (train_idx, valid_idx) in enumerate(outer_cv.split(X, y), start=1):
    X_outer_train = X.iloc[train_idx]
    X_outer_valid = X.iloc[valid_idx]
    y_outer_train = y.iloc[train_idx]
    y_outer_valid = y.iloc[valid_idx]

    search = GridSearchCV(
        estimator=rbf_pipeline,
        param_grid=rbf_param_grid,
        scoring="accuracy",
        cv=inner_cv,
        n_jobs=-1,
        refit=True,
    )
    search.fit(X_outer_train, y_outer_train)

    pred = search.predict(X_outer_valid)
    score = accuracy_score(y_outer_valid, pred)

    rbf_outer_scores.append(score)
    rbf_outer_params.append(search.best_params_)

    print(
        f"Fold {fold}: accuracy={score:.3f}, "
        f"C={search.best_params_['model__C']}, "
        f"gamma={search.best_params_['model__gamma']}"
    )

print(
    f"\nNested CV accuracy: "
    f"{np.mean(rbf_outer_scores):.3f} "
    f"(± {np.std(rbf_outer_scores):.3f})"
)

## 12. Model comparison

In [ ]:
results = pd.DataFrame(
    {
        "Model": ["Linear SVM", "RBF SVM"],
        "Test accuracy": [
            accuracy_score(y_test, linear_pred),
            accuracy_score(y_test, rbf_pred),
        ],
        "Nested CV mean accuracy": [
            np.mean(linear_outer_scores),
            np.mean(rbf_outer_scores),
        ],
        "Nested CV std": [
            np.std(linear_outer_scores),
            np.std(rbf_outer_scores),
        ],
    }
)

results

## Conclusions

This notebook compares linear and nonlinear Support Vector Machines on a balanced subset of the credit-card default dataset.

When interpreting the results, keep in mind that:

1. the modelling sample is artificially balanced;
2. accuracy therefore refers to that balanced experimental setup;
3. nested cross-validation gives a more robust estimate than a single train/test split;
4. preprocessing is fitted within each cross-validation fold to avoid data leakage.

A natural next step would be to evaluate the models on the **original class distribution** and compare accuracy with metrics such as **recall, F1-score, ROC-AUC, and PR-AUC**.
